In [1]:
import pandas as pd

In [52]:
df = pd.read_csv("../data/raw/NVDA/20260323_NVDA.csv")
df.tail()

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol
17229462,1774310398912591652,1774310398912428934,160,2,11667,C,A,188000000000,200,0,354734535,128,162718,1469795841,NVDA
17229463,1774310399045814419,1774310399045651504,160,2,11667,C,B,170000000000,1,0,966100195,128,162915,1469796874,NVDA
17229464,1774310399278178451,1774310399278015563,160,2,11667,C,A,196000000000,100,0,484308747,128,162888,1469797485,NVDA
17229465,1774310399407123334,1774310399406960412,160,2,11667,C,A,184000000000,40,0,693242499,128,162922,1469797680,NVDA
17229466,1774310399458248406,1774310399458085183,160,2,11667,C,B,171500000000,7,0,903599627,128,163223,1469797737,NVDA


In [90]:
df[(df["ts_recv"]<=1774252800159940542)&(df["side"]=="B")]

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol
5,1774252800159784921,1774252800159618849,160,2,11667,T,B,170290000000,100,0,0,0,166072,312259,NVDA
9,1774252800159866025,1774252800159702287,160,2,11667,T,B,170300000000,12,0,0,0,163738,312262,NVDA
13,1774252800159940542,1774252800159776735,160,2,11667,T,B,170300000000,12,0,0,0,163807,312268,NVDA


In [56]:
df_t = df.groupby("ts_recv")

KeyError: 'Column not found: 0'

In [46]:
df[(df["sequence"]==312259) & (df["action"]=="F")]

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol
6,1774252800159784921,1774252800159618849,160,2,11667,F,A,170290000000,100,0,29863,0,166072,312259,NVDA


In [14]:
df[(df["order_id"]==29863) & (df["ts_recv"]<=1774252800159866025)].iloc[-1]

ts_recv          1774252800159784921
ts_event         1774252800159618849
rtype                            160
publisher_id                       2
instrument_id                  11667
action                             C
side                               A
price                   170290000000
size                             100
channel_id                         0
order_id                       29863
flags                            128
ts_in_delta                   166072
sequence                      312259
symbol                          NVDA
Name: 7, dtype: object

In [82]:
df = df.iloc[:1000]

In [68]:
event_log = []
"""
event_log columns
    ts: time
    side: bid(B)/ask(A)
    price
    size
    event_type
"""
last_seen = {}
for temp_order in df.itertuples(index=False):
    # Trade, Fill, or None: no change
    if temp_order.action in ("T", "F", "N"):
        continue

    # Clear book: remove all resting orders
    if temp_order.action == "R":
        current_event = [temp_order.ts_recv,
                        temp_order.side,
                        temp_order.price,
                        temp_order.size,
                        "Clear"
                        ]

    # Add: insert a new order
    elif temp_order.action == "A":
        # For top-of-book publishers, remove previous order associated with this side
        # Modification of best ask/bid
        if temp_order.flags==64:
            current_event = [temp_order.ts_recv,
                            temp_order.side,
                            temp_order.price,
                            temp_order.size,
                            "Modify best"
                        ]
            # UNDEF_PRICE indicates the price level was removed. There's no new level to add
            if pd.isna(temp_order["price"]):
                continue
        else:
            current_event = [temp_order.ts_recv,
                        temp_order.side,
                        temp_order.price,
                        temp_order.size,
                        "Add"
                        ]

    # Cancel: partially or fully cancel some size from a resting order
    elif temp_order.action == "C":
        if temp_order.flags == 128:
            sequence = temp_order.sequence
            
            sequence_to_action = df.set_index("sequence").action.to_dict()
            if sequence_to_action.get(sequence) in {"T", "F"}:
                event_type = "Trade"
            else:
                event_type = "Cancel"
                
        current_event = [temp_order.ts_recv,
                        temp_order.side,
                        temp_order.price,
                        temp_order.size,
                        event_type
                        ]

    # Modify: change the price and/or size of a resting order
    elif temp_order.action == "M":
        
        previous = last_seen.get(temp_order.order_id)
        if previous is None:
            # Never seen this order before — can't compute delta
            # This can happen if the Add arrived before your data window
            continue
        else:
            delta_price = previous.price - temp_order.price
            delta_size  = previous.size  - temp_order.size
        current_event = [temp_order.ts_recv,
                        temp_order.side,
                        delta_price,
                        delta_size,
                        "Modify"
                        ]
    

    event_log.append(current_event)

event_log_df = pd.DataFrame(event_log, columns=["ts", "side", "price", "size", "event_type"])

In [74]:
ask_side = event_log_df[event_log_df["side"] == "A"]
best_ask = ask_side.groupby("ts")["price"].min()

In [77]:
best_ask

ts
1774252800021524740    202000000000
1774252800021527775    198000000000
1774252800051860436    180000000000
1774252800159752011    170290000000
1774252800159784921    170290000000
                           ...     
1774252808170773379    174900000000
1774252808908789832    175000000000
1774252809615093723    190000000000
1774252809960076059    185000000000
1774252810596087934    171110000000
Name: price, Length: 330, dtype: int64

In [79]:
best_ask[0]=100
best_ask

ts
1774252800021524740    202000000000
1774252800021527775    198000000000
1774252800051860436    180000000000
1774252800159752011    170290000000
1774252800159784921    170290000000
                           ...     
1774252808908789832    175000000000
1774252809615093723    190000000000
1774252809960076059    185000000000
1774252810596087934    171110000000
0                               100
Name: price, Length: 331, dtype: int64

In [91]:
def _best_price(df: pd.DataFrame):
    best_ask_series = {}
    best_bid_series = {}

    current_ask_prices = {}   # price -> size
    current_bid_prices = {}   # price -> size
    ask_book_series = {}   # ts -> full dict snapshot
    bid_book_series = {}   # ts -> full dict snapshot
    
    create_ask = []
    create_bid = []

    # Create aggregation state
    pending_create = None   # {"side": ..., "price": ..., "size": ..., "ts": ...}
    prev_ts = None   # track previous timestamp

    def _flush_create():
        """Apply the accumulated create event to the book."""
        if pending_create is None:
            return
        if pending_create["side"] == "A":
            current_ask_prices[pending_create["price"]] = pending_create["size"]
            create_ask.append([ts, pending_create["price"], pending_create["size"]])
        else:
            current_bid_prices[pending_create["price"]] = pending_create["size"]
            create_bid.append([ts, pending_create["price"], pending_create["size"]])
            
    def _snapshot(ts):
        """Record full book state and best prices at this timestamp."""
        best_ask_series[ts] = min(current_ask_prices) if current_ask_prices else None
        best_bid_series[ts] = max(current_bid_prices) if current_bid_prices else None
        # Store a copy — not a reference — so future updates don't mutate it
        ask_book_series[ts] = dict(current_ask_prices)
        bid_book_series[ts] = dict(current_bid_prices)


    for row in df.itertuples(index=False):
        ts = row.ts

        # When timestamp changes, snapshot the completed previous timestamp
        # But only if no Create burst is pending (book not fully updated yet)
        if prev_ts is not None and ts != prev_ts and pending_create is None:
            _snapshot(prev_ts)
            
        
        # Check if this row continues an ongoing Create burst
        # A burst continues only if same side, same price, and still an Add
        is_create_continuation = (
            pending_create is not None
            and row.event_type == "Add"
            and row.side == pending_create["side"]
            and row.price == pending_create["price"]
        )

        if is_create_continuation:
            # Accumulate into the pending create
            pending_create["size"] += row.size
            pending_create["ts"] = ts
            # Do not apply to book yet — keep accumulating

        else:
            # This row breaks the burst — flush whatever was pending
            _flush_create()
            pending_create = None

            # Now handle the current row normally
            if row.event_type == "Modify best":
                # F_TOB update — replace the entire side's best level
                if row.side == "A":
                    # Remove old best ask entry and write new one
                    if current_ask_prices:
                        old_best = min(current_ask_prices)
                        current_ask_prices.pop(old_best, None)
                    if row.price is not None:
                        current_ask_prices[row.price] = row.size
                else:
                    if current_bid_prices:
                        old_best = max(current_bid_prices)
                        current_bid_prices.pop(old_best, None)
                    if row.price is not None:
                        current_bid_prices[row.price] = row.size

            elif row.event_type == "Add":
                # Check if this is the start of a new Create burst
                # A Create is an Add that places a new best level
                # i.e. ask price below current best ask, or bid price above current best bid
                is_new_best_ask = (
                    row.side == "A"
                    and current_ask_prices
                    and row.price < min(current_ask_prices)
                )
                is_new_best_bid = (
                    row.side == "B"
                    and current_bid_prices
                    and row.price > max(current_bid_prices)
                )

                if is_new_best_ask or is_new_best_bid:
                    # Start accumulating a new Create burst
                    pending_create = {
                        "side":  row.side,
                        "price": row.price,
                        "size":  row.size,
                        "ts":    ts,
                    }
                    # Do not apply to book yet
                else:
                    # Regular add at an existing or non-best level
                    if row.side == "A":
                        current_ask_prices[row.price] = current_ask_prices.get(row.price, 0) + row.size
                    else:
                        current_bid_prices[row.price] = current_bid_prices.get(row.price, 0) + row.size

            elif (row.event_type == "Cancel") or (row.event_type == "Trade"):
                if row.side == "A":
                    current_ask_prices[row.price] -= row.size
                    # Only remove the level entirely if no shares remains
                    if current_ask_prices[row.price] <= 0:
                        current_ask_prices.pop(row.price)
                else:
                    current_bid_prices[row.price] -= row.size
                    if current_bid_prices[row.price] <= 0:
                        current_bid_prices.pop(row.price)

            elif row.event_type == "Clear":
                current_ask_prices.clear()
                current_bid_prices.clear()
            
        prev_ts = ts

        # Snapshot current best after every row
        # Skip snapshot during an ongoing Create burst — book not updated yet
        if pending_create is None:
            best_ask_series[ts] = min(current_ask_prices) if current_ask_prices else None
            best_bid_series[ts] = max(current_bid_prices) if current_bid_prices else None

    # Flush any pending create at end of data
    _flush_create()
    # Snapshot the final timestamp
    if prev_ts is not None:
        _snapshot(prev_ts)

    best_ask = pd.Series(best_ask_series)
    best_bid = pd.Series(best_bid_series)
    
    create_ask_df = pd.DataFrame(create_ask, columns=["ts", "price", "size"])
    create_bid_df = pd.DataFrame(create_bid, columns=["ts", "price", "size"])

    return best_ask, best_bid, ask_book_series, bid_book_series, create_ask_df, create_bid_df

In [99]:
best_ask, best_bid, ask_book_series, bid_book_series, create_ask_df, create_bid_df = _best_price(event_log_df)
best_bid.iloc[200:210]

1774252802127288028    1.707600e+11
1774252802129424487    1.707600e+11
1774252802132962852    1.707600e+11
1774252802135593605    1.707600e+11
1774252802135730881    1.707600e+11
1774252802163765676    1.707600e+11
1774252802224716519    1.707600e+11
1774252802224764837    1.707600e+11
1774252802227260011    1.707600e+11
1774252802257563559    1.707600e+11
dtype: float64

In [101]:
best_bid[1774252802135593605]

np.float64(170760000000.0)

In [106]:
pd.Series(bid_book_series[1774252802135593605]).sort_index()

155000000000     20
157200000000      2
164410000000     10
165000000000    809
165960000000     50
166160000000      2
167000000000      1
167880000000     20
168000000000      5
168870000000     19
170000000000    569
170010000000     39
170100000000    800
170200000000    800
170300000000    800
170500000000      1
170540000000    501
170760000000     34
dtype: int64

In [ ]:
def grid_order_book(df:pd.DataFrame, tick: float) -> pd.DataFrame:
    for